# CardioBreath — Google Colab / Kaggle T4 GPU Training Notebook

Trains all **5 subject-disjoint cross-validation folds** for:
1. **Baseline 1:** `HRV + LightGBM`
2. **Baseline 2:** `Raw 1-min ECG 1D-CNN`
3. **Ablations:** `RRI-Only`, `EDR-Only`, `Dual-Stream 1-min`
4. **Proposed Model:** `CardioBreathNet` (Dual-Stream Multi-Scale 1D-CNN + BiGRU + Temperature Scaling)

Then exports the 5-fold ensemble to **ONNX**, runs the **locked held-out evaluation (`x01–x35`)**, and zips all trained artifacts for download back into your local repo.

> **Runtime:** Select `Runtime -> Change runtime type -> T4 GPU`. Total wall-clock time is ~15–20 minutes.

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
# 1. Clone CardioBreath repository (phase_2 branch)
REPO_URL = 'https://github.com/Hackeddog/CardioBreath_AI_for_medicine_hackathon.git'
BRANCH = 'phase_2'

!rm -rf CardioBreath_AI_for_medicine_hackathon
!git clone --branch {BRANCH} {REPO_URL}
%cd CardioBreath_AI_for_medicine_hackathon
!pip install -q -r requirements.txt

In [ ]:
# 2. Download PhysioNet Apnea-ECG (70 nights + 8 resp-belt nights + Challenge 2000 test labels)
!python scripts/download_data.py --workers 8

# 3. Run biomedical DSP preprocessing + physical EDR validation
!python scripts/preprocess.py

In [ ]:
# 4. Train all baselines, ablations, and CardioBreathNet across all 5 subject-disjoint folds
!python train.py --model all --device cuda --seed 42

In [ ]:
# 5. Export CardioBreath 5-fold ensemble to ONNX (+ INT8) and benchmark CPU latency
!python scripts/export_onnx.py --model cardiobreath --quantize

In [ ]:
# 6. Evaluate 5-fold Cross-Validation first, then run SINGLE locked evaluation on x01-x35
!python evaluate.py --split val --seed 42 --device cuda
!python evaluate.py --split test --seed 42 --device cuda

print(open('reports/benchmark_table_test.md').read())

In [ ]:
# 7. Package trained checkpoints, ONNX weights, and evaluation figures for download
!zip -r cardiobreath_trained_artifacts.zip artifacts/onnx artifacts/checkpoints/lgbm artifacts/checkpoints/cardiobreath reports
from google.colab import files
files.download('cardiobreath_trained_artifacts.zip')